<small><b>02 Vector Search</b> — Step-by-step: load cleaned data → Documents + metadata → CharacterTextSplitter → Embeddings → Chroma → Top-K retriever. Default embeddings: <code>Qwen/Qwen3-Embedding-0.6B</code> (<code>EMBEDDING_PROVIDER=qwen</code>). Alternatives: <code>voyage</code> / <code>openai</code> / <code>local</code>.</small>

<small><b>Step 0 — Setup</b>: import helpers. Default model <code>Qwen/Qwen3-Embedding-0.6B</code> (local; first run downloads ~1GB). Optional <code>HF_TOKEN</code> for faster Hub downloads. Chroma path is model-specific.</small>

In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.vector_search import (
    CLEANED_PATH,
    build_chroma,
    build_documents,
    format_hits,
    get_embeddings,
    load_cleaned,
    resolve_chroma_dir,
    search,
)

# Use a small sample while learning (set to None for all ~5.6k rows)
SAMPLE_SIZE = 500
TOP_K = 5
CHROMA_DIR = resolve_chroma_dir()

print("EMBEDDING_PROVIDER:", os.getenv("EMBEDDING_PROVIDER"))
print("QWEN_EMBEDDING_MODEL:", os.getenv("QWEN_EMBEDDING_MODEL"))
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))
print("CLEANED_PATH:", CLEANED_PATH)
print("CHROMA_DIR:", CHROMA_DIR)
print("SAMPLE_SIZE:", SAMPLE_SIZE)

EMBEDDING_PROVIDER: qwen
QWEN_EMBEDDING_MODEL: Qwen/Qwen3-Embedding-0.6B
HF_TOKEN set: True
CLEANED_PATH: C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_cleaned.csv
CHROMA_DIR: C:\Users\suxia\Desktop\Saas-Recommender(2026)\.chroma\saas_products_qwen_Qwen_Qwen3-Embedding-0.6B
SAMPLE_SIZE: 500


<small><b>Step 1 — Load</b> <code>data/saas_cleaned.csv</code>. Embedding input is the <code>searchable_text</code> column from notebook 01.</small>

In [2]:
df = load_cleaned(CLEANED_PATH, sample_size=SAMPLE_SIZE)
print("rows:", len(df))
display(df[["name", "tagline", "topics", "votes_count", "searchable_text"]].head(3))
print("\nsearchable_text length (chars):")
display(df["searchable_text"].str.len().describe())

rows: 500


,name,tagline,topics,votes_count,searchable_text
0,Bluedot 2.1,Record on Apple Watch. Sync with Claude,Productivity,247,Bluedot 2.1. Record on Apple Watch. Sync with ...
1,Powabase,"Build AI apps with Postgres, RAG, and agents","AI, Developer Tools",223,"Powabase. Build AI apps with Postgres, RAG, an..."
2,Oasis Browser for Mac,A privacy-first AI browser you can train anony...,"AI, Productivity",173,Oasis Browser for Mac. A privacy-first AI brow...



searchable_text length (chars):


count    500.0000
mean     452.9000
std      122.8683
min      172.0000
25%      348.0000
50%      482.5000
75%      563.0000
max      636.0000
Name: searchable_text, dtype: float64

<small><b>Step 2 — Documents + metadata</b>. Each product becomes a LangChain <code>Document</code>. Metadata keeps name/tagline/votes/topics/platforms + engineered flags (source has no website_url / product_hunt_url).</small>

In [3]:
from src.vector_search import row_to_document

example = row_to_document(df.iloc[0])
print("page_content (first 300 chars):\n", example.page_content[:300], "...\n")
print("metadata keys:", sorted(example.metadata.keys()))
print("metadata sample:", {k: example.metadata[k] for k in ["name", "tagline", "votes_count", "topics", "platforms"]})

page_content (first 300 chars):
 Bluedot 2.1. Record on Apple Watch. Sync with Claude. Bluedot 2.1 brings your real-world conversations into Claude. Record conversations directly from your Apple Watch, then sync them with Claude through MCP. Capture customer calls, hallway chats, interviews, coffee meetings, and in-person conversat ...

metadata keys: ['comments_count', 'daily_rank_clean', 'engagement_ratio', 'id', 'is_ai_product', 'is_dev_tool', 'is_productivity', 'is_saas_product', 'is_viral', 'log_votes', 'main_category', 'name', 'platforms', 'tagline', 'topics', 'votes_count', 'weekly_rank_clean']
metadata sample: {'name': 'Bluedot 2.1', 'tagline': 'Record on Apple Watch. Sync with Claude', 'votes_count': 247, 'topics': 'Productivity', 'platforms': 'Website'}


<small><b>Step 3 — CharacterTextSplitter</b>. Splits long descriptions; short Product Hunt blurbs usually stay as one chunk.</small>

In [4]:
documents = build_documents(df, chunk_size=1000, chunk_overlap=100)
print("products:", len(df), "→ chunks:", len(documents))
print("first chunk length:", len(documents[0].page_content))

products: 500 → chunks: 500
first chunk length: 498


<small><b>Step 4 — Embeddings</b>. Current default: <code>Qwen/Qwen3-Embedding-0.6B</code> (1024-d, local). Queries use Qwen's built-in <code>prompt_name="query"</code>. Changing the model means you must rebuild Chroma (Step 5).</small>

In [5]:
# provider=None reads EMBEDDING_PROVIDER from .env (default: local)
embeddings = get_embeddings(provider=None)

# Smoke-test: one short string → vector
vec = embeddings.embed_query("AI coding agent for solo founders")
print("embedding dim:", len(vec))
print("first 8 dims:", [round(x, 4) for x in vec[:8]])

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

embedding dim: 1024
first 8 dims: [0.0933, 0.0199, -0.0048, -0.0583, -0.0542, -0.1138, 0.0076, 0.0147]


<small><b>Step 5 — Chroma</b>. Persist vectors under a model-specific folder (see <code>CHROMA_DIR</code>). If you already built the index from the terminal, set <code>rebuild=False</code> to load it instead of embedding 500 docs again.</small>

In [6]:
from src.vector_search import load_chroma, get_chroma_dir

CHROMA_DIR = get_chroma_dir()
# Use False if index already exists (e.g. built via terminal) — avoids re-embedding ~15–20 min on CPU
REBUILD = False

print("CHROMA_DIR:", CHROMA_DIR)
if REBUILD:
    print("Building index (this can take a while on CPU)...")
    vectorstore = build_chroma(
        documents,
        embeddings=embeddings,
        persist_directory=CHROMA_DIR,
        rebuild=True,
    )
else:
    print("Loading existing index from disk...")
    vectorstore = load_chroma(embeddings=embeddings, persist_directory=CHROMA_DIR)

print("collection count:", vectorstore._collection.count())
print("Step 5 done.")

CHROMA_DIR: C:\Users\suxia\Desktop\Saas-Recommender(2026)\.chroma\saas_products_qwen_Qwen_Qwen3-Embedding-0.6B
Loading existing index from disk...
collection count: 500
Step 5 done.


<small><b>Step 6 — Retriever</b>. Natural-language query → embed query → cosine similarity → Top-K products.</small>

In [7]:
from src.vector_search import get_chroma_dir, load_chroma

print("Step 6 starting...", flush=True)

# If Step 5 wasn't run in this kernel, load the on-disk Qwen index
if "vectorstore" not in globals() or vectorstore is None:
    print("vectorstore missing — loading from disk (model load may take 30–90s)...", flush=True)
    vectorstore = load_chroma(persist_directory=get_chroma_dir())

print("collection count:", vectorstore._collection.count(), flush=True)

queries = [
    "I need a tool that auto-generates weekly reports and integrates with Slack",
    "AI coding agent for solo founders",
    "low-cost freemium SaaS for collecting customer feedback",
]

for q in queries:
    print("=" * 72, flush=True)
    print("QUERY:", q, flush=True)
    print("-" * 72, flush=True)
    hits = search(q, k=TOP_K, vectorstore=vectorstore)
    print(format_hits(hits), flush=True)
    print(flush=True)

print("Step 6 done.", flush=True)

Step 6 starting...
collection count: 500
QUERY: I need a tool that auto-generates weekly reports and integrates with Slack
------------------------------------------------------------------------
1. Octolane — Self-driving AI CRM that you can talk to
   votes=126 | topics=General | category=General | platforms=Twitter, Website
   snippet: Octolane. Self-driving AI CRM that you can talk to. Octolane is chat-first Self-driving AI CRM: say "follow up with David" or "show me stuck deals" and it does the thing. self-driv...

2. Cronmint — The cron monitor indie devs actually enjoy using
   votes=4 | topics=SaaS, Developer Tools, Productivity | category=SaaS | platforms=Website, Twitter
   snippet: Cronmint. The cron monitor indie devs actually enjoy using. Schedule HTTP requests. Get email alerts when they break. Free for 5 jobs. $9/mo for everything else.Schedule any HTTP r...

3. HookDeploy — Inspect, transform, replay & forward webhooks in real time.
   votes=2 | topics=Developer Tools |

<small><b>Step 7 — Your turn</b>. Change the query below. When results look good, set <code>SAMPLE_SIZE = None</code> in Step 0 and re-run Steps 1→5 to index the full dataset.</small>

In [8]:
my_query = "privacy-first AI browser for Mac"
hits = search(my_query, k=TOP_K, vectorstore=vectorstore)
print(format_hits(hits))

1. Oasis Browser for Mac — A privacy-first AI browser you can train anonymously
   votes=173 | topics=AI, Productivity | category=AI | platforms=Instagram, Twitter, Website
   snippet: Oasis Browser for Mac. A privacy-first AI browser you can train anonymously. Oasis is a refuge from noisy, scattered browsing. Privacy comes first, in an elegant experience that AI...

2. Tap — A Mac browser that keeps everything you open side by side.
   votes=4 | topics=AI, Productivity | category=AI | platforms=Website
   snippet: Tap. A Mac browser that keeps everything you open side by side.. Most browsers, even the new AI ones, are just tabs with a chat sidebar bolted on. Tap changes the model itself. Eve...

3. brew-browser — The Homebrew GUI Mac users have been waiting for.
   votes=4 | topics=Developer Tools | category=Developer Tools | platforms=Github, Website
   snippet: brew-browser. The Homebrew GUI Mac users have been waiting for.. Homebrew has 13,000+ packages and zero great GUIs. brew-br